<br>

# Demo I: Loading Pre-computed Embeddings (word embeddings)


In this demo, we load **pre-computed word embeddings** (GloVe) — no neural network involved, just a lookup table mapping words to vectors — and explore how word similarity works in that vector space.

In [1]:
# %pip install gensim

In [2]:
import gensim
import gensim.downloader as api

In [3]:
# a large word2vec model
# model_large = api.load("word2vec-google-news-300")

# note: if that model is too large to load, you can use the one below
model = api.load("glove-wiki-gigaword-100")

**Note:** You'll see the GloVe object referred to as a `model` in code (e.g., `model = api.load(...)`), and it's common to call pre-trained embeddings a "model" loosely, since they represent a learned mapping of words to vectors.

But technically, we're **not** loading a neural network — we're loading a pre-computed **lookup table**: a large dictionary mapping each word to its vector. The variable name `model` is just a naming convention, not a reflection of what's actually loaded.


GloVe was trained on Wikipedia (2014 snapshot) and Gigaword, a dataset of about 4 billion words from decades of news articles (e.g., New York Times, Associated Press). Together, these corpora provided a massive, diverse text source for learning word co-occurrence statistics.

Note: GloVe didn't use a neural network — it was built using a **co-occurrence matrix** (how often words appear near each other) and, which then was factorized to produce vectors. After training, this matrix and the factorization process were discarded. What was saved and distributed are just the **resulting word vectors** — one per word in the vocabulary.

<br>

In [4]:
# let's check how many embeddings (words) we're loading
print(len(model.key_to_index))

400000


In [5]:
# each word has an encoding
model['boat']

array([-0.0083229,  0.3696   , -0.24436  , -0.89288  , -0.23607  ,
       -0.41659  ,  0.48309  ,  0.91159  ,  0.14498  , -0.096963 ,
        0.55061  ,  1.0376   ,  0.32243  , -0.01381  ,  0.0098052,
       -0.66346  ,  0.27949  , -0.74099  , -0.29602  ,  0.64596  ,
        1.1305   ,  0.54629  ,  0.49664  , -0.87378  ,  0.42399  ,
        0.35015  , -1.9151   ,  0.010363 ,  0.35684  , -0.32398  ,
       -0.66927  ,  0.43628  , -0.20924  ,  0.28862  ,  0.63752  ,
       -0.18789  , -0.079442 ,  0.30494  ,  0.8829   , -0.3143   ,
       -1.2595   , -0.72301  ,  0.077278 , -0.045894 ,  1.0251   ,
        0.25472  , -0.2566   ,  0.18428  ,  0.34037  ,  0.53185  ,
       -0.070906 ,  0.57464  ,  0.5131   ,  1.1666   ,  0.1848   ,
       -1.4466   , -0.41846  ,  0.011812 ,  2.1553   ,  0.52012  ,
       -0.9029   ,  0.43183  ,  0.60584  ,  0.72845  , -0.32243  ,
        0.73929  , -0.68845  ,  0.25407  , -0.20834  , -0.059242 ,
       -0.45655  , -0.27773  ,  0.7168   ,  0.075051 ,  0.4167

In [6]:
# and you can search for "similar" words (words that could occupy the same space)
model.most_similar('boat')

[('boats', 0.8819124102592468),
 ('ship', 0.8258435726165771),
 ('vessel', 0.8200733661651611),
 ('ferry', 0.77042156457901),
 ('sailing', 0.7610523700714111),
 ('ships', 0.7407434582710266),
 ('capsized', 0.7383140921592712),
 ('barge', 0.7286107540130615),
 ('fishing', 0.7228102087974548),
 ('yacht', 0.7208371162414551)]

In [7]:
model.most_similar('book')

[('books', 0.847648561000824),
 ('novel', 0.8181166648864746),
 ('published', 0.8023924231529236),
 ('story', 0.7941390872001648),
 ('author', 0.7937875390052795),
 ('wrote', 0.7930577397346497),
 ('essay', 0.7821518182754517),
 ('biography', 0.7754694819450378),
 ('written', 0.760090172290802),
 ('fiction', 0.7549652457237244)]

In [8]:
model.most_similar('king')

[('prince', 0.7682329416275024),
 ('queen', 0.7507690787315369),
 ('son', 0.7020888328552246),
 ('brother', 0.6985775828361511),
 ('monarch', 0.6977890133857727),
 ('throne', 0.691999077796936),
 ('kingdom', 0.6811409592628479),
 ('father', 0.680202841758728),
 ('emperor', 0.6712858080863953),
 ('ii', 0.6676074266433716)]

<br>

## Now for the really cool part...

In [9]:
# you can "add" or subtract tendencies
model.most_similar(positive=['woman','king'],negative=['man'],topn=1)

[('queen', 0.7698540687561035)]

In [10]:
model.most_similar(positive=['woman','boy'],negative=['man'],topn=1)

[('girl', 0.9095936417579651)]

In [11]:
#if at first you don't succeed
model.most_similar(positive=['grape','beer'],negative=['barley'],topn=1)

[('wine', 0.7183154225349426)]

In [12]:
model.most_similar(positive=['lisbon','spain'],negative=['madrid'],topn=1)

[('portugal', 0.8062521815299988)]

<br>

## How similar are two words?

One way to measure similarity between two word vectors is **cosine similarity**.

Think of each word's embedding as an arrow pointing in some direction in space. Cosine similarity measures the **angle between two arrows**:

- If two arrows point in nearly the **same direction** → angle is small → similarity is **close to 1** (very similar)
- If two arrows point in **opposite directions** → similarity is **close to -1** (opposite meaning)
- If the arrows are **perpendicular** (unrelated) → similarity is **close to 0**

The key insight is that it doesn't matter how long the arrows are — only the direction matters. So `king` and `monarch` should have a small angle between them (high similarity), while `king` and `duck` should not.

<br>

![embeddings_cosine_similarity_2](../_images/embeddings_cosine_similarity_2.png)

<br>

![embeddings_cosine_similarity_1](../_images/embeddings_cosine_similarity_1.jpg)


In [13]:
# why does this work?
from sklearn.metrics.pairwise import cosine_similarity

cosine_similarity([model['queen']],[model['duck']])

array([[0.24254493]], dtype=float32)

In [14]:
cosine_similarity([model['queen']],[model['woman']])

array([[0.5095154]], dtype=float32)

In [15]:
cosine_similarity([model['queen']],[model['crown']])

array([[0.66805613]], dtype=float32)

In [16]:
cosine_similarity([model['queen']],[model['monarch']])

array([[0.6683258]], dtype=float32)

In [17]:
# now that we established a baseline for similarity...

In [18]:
cosine_similarity([model['king']-model['queen']],[model['son']-model['daughter']])

array([[0.7017219]], dtype=float32)

In [19]:
cosine_similarity([model['france'] - model['paris']], [model['germany'] - model['berlin']])

array([[0.7823462]], dtype=float32)

<br>

# Demo II: Loading a Neural Network (sentence embeddings)


In this demo **we will load a full neural network** — unlike the previous demo, where we only loaded a table of pre-computed embeddings.

Also, we move from **word embeddings** (vectors for single words) to **sentence embeddings** (vectors for full sentences).


`SentenceTransformer('all-MiniLM-L6-v2')` loads a pre-trained, fine-tuned model. When we call `model.encode(sentences)`, it **runs inference**: each sentence is passed through the network and a vector is computed on the fly. This means:

- There is no lookup table — nothing is pre-computed
- The same word can produce **different vectors** depending on context (contextual embeddings)
- We get **sentence-level** vectors, not word-level vectors

We will:
- Load a pre-trained `SentenceTransformer` model (`all-MiniLM-L6-v2`)
- Encode sentences into dense vectors that capture overall meaning
- Compare a query against multiple documents using **cosine similarity**
- Test a harder example with ambiguous words to see semantic understanding in action


<br>


In [20]:
# %pip install sentence_transformers

In [21]:
from sentence_transformers import SentenceTransformer

# Load a pre-trained Sentence Transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Example sentences
sentences = [
        "A dog is playing with a ball.",
        "A cat is sitting on the mat."
    ]
embeddings = model.encode(sentences)

# Print embeddings
print("Embedding shape:", embeddings.shape)

Embedding shape: (2, 384)


In [22]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# A query from a customer
query = "Can I get my money back if I don't like the product?"

# A collection of documents from the FAQs
documents = [
    "Our return policy allows refunds within 30 days of purchase",
    "We offer free shipping on orders over 50€",
    "International orders typically arrive within 7-14 business days",
    "Track your order using the tracking number sent via email",
]

# Encode the query and documents
query_embedding = model.encode([query])
doc_embeddings = model.encode(documents)

# Compute cosine similarities
similarities = cosine_similarity(query_embedding, doc_embeddings)

for index, score in enumerate(similarities[0]):
    print(f"Sentence {index} → Similarity... {score}")

Sentence 0 → Similarity... 0.4892561435699463
Sentence 1 → Similarity... 0.17404179275035858
Sentence 2 → Similarity... 0.006786830723285675
Sentence 3 → Similarity... 0.128294438123703
